# Sensitivity Analysis of Latent Dimensions for SAMVAE

This notebook analyzes how different latent dimension configurations affect model performance (CI-IBS metric) across modalities in the hyperparameter optimization results.

For each dataset, analysis type, and modality combination, the notebook:

1. Extracts hyperparameter optimization results from the results directory

2. Filters latent dimensions of interest:

   - **Clinical modality**: 5, 10. The boxplots display latent dimensions on the x-axis and CI-IBS performance on the y-axis, with different colors representing different modality combinations.

   - **Multimodal combinations**: 5, 50, 500 (for the omic/image dimension)

3. Selects the **top N best performing configurations** (N=3 by default) based on CI-IBS metric. Generates boxplots showing the distribution of these top N values across different latent dimensions

## 1. Configuration

In [84]:
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
from typing import Tuple, List

# User-editable configuration 
SCRIPT_CONFIG = {
    "results_dir": os.path.join("..", "results", "Hyperparameter_optimization"),
    "output_dir": os.path.join("figs", "sensitive_analysis_samvae"),
    "datasets": ["brca", "lgg"],
    "analyses": ["Survival_Analysis", "Competing_Risks"],
    "top_n": 3 # Define how many top values to display here
}

## 2. Helper Functions

In [85]:
def find_result_table_path(base_dir: str, analysis_type: str) -> str:
    candidates = []
    if analysis_type == "Competing_Risks":
        candidates = [
            os.path.join(base_dir, "total_results_table_cr.tex"),
            os.path.join(base_dir, "total_results_table_cr.txt"),
            os.path.join(base_dir, "results_table_1.txt"),
        ]
    else:
        candidates = [
            os.path.join(base_dir, "total_results_table.tex"),
            os.path.join(base_dir, "total_results_table.txt"),
            os.path.join(base_dir, "total_results_table_2.txt"),
        ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return ""


def parse_ascii_table(path: str) -> Tuple[List[str], List[List[str]]]:
    headers: List[str] = []
    rows: List[List[str]] = []
    if not path:
        return headers, rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip("\n")
            if not line.startswith("|"):
                continue
            parts = [c.strip() for c in line.split("|")]
            parts = [p for p in parts if p]
            if not parts:
                continue
            if parts[0] == "DATASETS" or parts[0].upper() == "DATASETS":
                headers = parts
            elif parts[0] == "+------------+":
                continue
            elif headers and len(parts) == len(headers):
                rows.append(parts)
    return headers, rows


def parse_tex_table(path: str) -> Tuple[List[str], List[List[str]]]:
    headers: List[str] = []
    rows: List[List[str]] = []
    if not path:
        return headers, rows
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or "&" not in line:
                continue
            # Remove latex line ending and extra slashes
            if line.endswith("\\\\"):
                line = line[:-2].strip()
            parts = [p.strip() for p in line.split("&")]
            # Identify header by presence of required columns
            if ("DATASETS" in parts and "MODES" in parts and "LATENT" in parts) or ("LATENT" in parts and "HIDDEN" in parts):
                headers = parts
                continue
            if headers and len(parts) >= len(headers):
                # Normalize row to header length
                parts = parts[:len(headers)]
                rows.append(parts)
    return headers, rows


_MEAN_STD_RE = re.compile(r"([0-9]*\.?[0-9]+)\s*±\s*([0-9]*\.?[0-9]+)")


def parse_value(v: str) -> float:
    v = v.strip()
    m = _MEAN_STD_RE.match(v)
    if m:
        return float(m.group(1))
    try:
        return float(v)
    except Exception:
        return float("nan")

## 3. Data Collection

In [87]:
# Collect all records from hyperparameter optimization results:
def collect_records(results_root: str) -> pd.DataFrame:
    records = []

    for analysis in ["Competing_Risks", "Survival_Analysis"]:
        at_dir = os.path.join(results_root, analysis)
        if not os.path.isdir(at_dir): continue
        
        for dataset in sorted(os.listdir(at_dir)):
            ds_dir = os.path.join(at_dir, dataset)
            if not os.path.isdir(ds_dir): continue
            
            for mode in sorted(os.listdir(ds_dir)):
                mode_dir = os.path.join(ds_dir, mode)
                if not os.path.isdir(mode_dir): continue
                
                table_path = find_result_table_path(mode_dir, analysis)
                if not table_path:
                    for sub in os.listdir(mode_dir):
                        cand = os.path.join(mode_dir, sub)
                        if os.path.isdir(cand):
                            p = find_result_table_path(cand, analysis)
                            if p: 
                                table_path = p; break

                if not table_path: continue
                headers, rows = parse_tex_table(table_path) if table_path.endswith('.tex') else parse_ascii_table(table_path)
                if not headers or not rows: continue

                hmap = {h: i for i, h in enumerate(headers)}
                
                # Process rows
                for r in rows:
                    try:
                        # Extract Latents
                        l_entry = r[hmap.get("LATENT", -1)]
                        latents = [int(float(x)) for x in l_entry.replace('[','').replace(']','').split(',')]
                        
                        # Extract Risk
                        risk = int(float(r[hmap["RISK"]])) if "RISK" in hmap else 0
                        
                        # Calculate CI - IBS
                        if "C-INDEX" in hmap and "IBS" in hmap:
                            val = parse_value(r[hmap["C-INDEX"]]) - parse_value(r[hmap["IBS"]])
                        elif "CI - IBS" in hmap:
                            val = parse_value(r[hmap["CI - IBS"]])
                        else:
                            continue

                        # Define which latent to use for the plot right here
                        # If clinical, use the first latent. If multimodal, the second.
                        target_latent = latents[0] if mode == "clinical" else (latents[1] if len(latents) > 1 else latents[0])
                        label_combo = f"{mode}_{target_latent}"

                        records.append({
                            "analysis": analysis,
                            "dataset": dataset,
                            "mode": mode,
                            "risk": risk,
                            "target_latent": target_latent,
                            "label": label_combo,
                            "ci_minus_ibs": val
                        })
                    except Exception: continue

    return pd.DataFrame(records)

## 4. Load Data

In [89]:
# Load all hyperparameter optimization results:
results_dir = os.path.abspath(SCRIPT_CONFIG["results_dir"])
df = collect_records(results_dir)

if df.empty:
    print("No records found. Please verify results_dir.")
else:
    print(f"\nDatasets: {sorted(df['dataset'].unique())}")
    print(f"Analyses: {sorted(df['analysis'].unique())}")
    print(f"Modes: {sorted(df['mode'].unique())}")
    
    # Show sample
    print("\nSample data:")
    display(df.head(10))


Datasets: ['brca', 'lgg']
Analyses: ['Competing_Risks', 'Survival_Analysis']
Modes: ['clinical', 'clinical_omic_RNAseq', 'clinical_omic_adn', 'clinical_omic_cnv', 'clinical_omic_miRNA', 'clinical_wsi_patches_1_patch']

Sample data:


,analysis,dataset,mode,risk,target_latent,label,ci_minus_ibs
0,Competing_Risks,brca,clinical,0,5,clinical_5,0.426
1,Competing_Risks,brca,clinical,1,5,clinical_5,0.390
2,Competing_Risks,brca,clinical,0,5,clinical_5,0.338
3,Competing_Risks,brca,clinical,1,5,clinical_5,0.457
4,Competing_Risks,brca,clinical,0,5,clinical_5,0.391
5,Competing_Risks,brca,clinical,1,5,clinical_5,0.425
6,Competing_Risks,brca,clinical,0,5,clinical_5,0.387
7,Competing_Risks,brca,clinical,1,5,clinical_5,0.385
8,Competing_Risks,brca,clinical,0,5,clinical_5,0.379
9,Competing_Risks,brca,clinical,1,5,clinical_5,0.403


## 5. Plotting Functions

In [90]:
# Function to generate a boxplot showing all modality-latent combinations
def plot_boxplots_all_combinations(df, output_dir, datasets=None, analyses=None, top_n=3):
    if df.empty: return
    
    # 1. Global filters
    if datasets: df = df[df['dataset'].isin(datasets)]
    if analyses: df = df[df['analysis'].isin(analyses)]
    
    # 2. Filter latent dimensions of interest (5, 10 for clinical; 5, 50, 500 for the rest) 
    is_clinical = df['mode'] == 'clinical'
    keep_clinical = is_clinical & df['target_latent'].isin([5, 10])
    keep_omic = (~is_clinical) & df['target_latent'].isin([5, 50, 500])
    df_filtered = df[keep_clinical | keep_omic].copy()

    # 3. Get the TOP N values
    df_top = (df_filtered.sort_values("ci_minus_ibs", ascending=False)
              .groupby(["analysis", "dataset", "risk", "label"])
              .head(top_n))

    # 4. Generate plots (Iterating by Dataset/Analysis/Risk combination)
    sns.set_style("whitegrid")
    
    unique_groups = df_top.groupby(["analysis", "dataset", "risk"])
    
    for (analysis, dataset, risk), group_data in unique_groups:
        
        # Sort to ensure clinical appears with [5] before [10]
        # Create a sort key: mode first, then target_latent
        group_data = group_data.copy()
        group_data['sort_key'] = group_data.apply(
            lambda row: (row['mode'], row['target_latent']), axis=1
        )
        group_data = group_data.sort_values('sort_key')
        
        # Create display names for legend 
        def format_modality_name(mode):
            if mode == 'clinical':
                return 'clinical'
            elif 'clinical_omic_' in mode:
                # Extract omic type and format
                omic_type = mode.replace('clinical_omic_', '')
                # Map specific names
                if omic_type == 'RNAseq':
                    return 'clinical + RNAseq'
                elif omic_type == 'adn':
                    return 'clinical + DNA'
                elif omic_type == 'cnv':
                    return 'clinical + CNV'
                elif omic_type == 'miRNA':
                    return 'clinical + miRNA'
                else:
                    return f'clinical + {omic_type}'
            elif 'patch' in mode.lower():
                return 'clinical + 1 patch'
            else:
                return mode
        
        group_data['display_mode'] = group_data['mode'].apply(format_modality_name)
        
        fig, ax = plt.subplots(figsize=(max(10, 0.5 * group_data['label'].nunique()), 6))
        
        analysis_abbr = "CR" if analysis == "Competing_Risks" else "SA"
        if analysis == "Competing_Risks":
            title = f"{dataset.upper()}-{analysis_abbr} - Sensitivity Analysis of Latent Dimensions (Risk {risk})"
        else:
            title = f"{dataset.upper()}-{analysis_abbr} - Sensitivity Analysis of Latent Dimensions"
        ax.set_title(title, fontsize=14, fontweight='bold')
        
        sns.boxplot(data=group_data, 
                    x="label", 
                    y="ci_minus_ibs", 
                    hue="display_mode", 
                    dodge=False, 
                    palette="tab10", 
                    order=group_data['label'].unique(), 
                    ax=ax)
        
        # Set latent dimensions only
        tick_positions = range(len(group_data['label'].unique()))
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([str(group_data[group_data['label']==lbl]['target_latent'].iloc[0]) 
                            for lbl in group_data['label'].unique()])
        
        ax.set_xlabel("Latent Dimension of Modalities", fontweight='bold')
        ax.set_ylabel("CI - IBS", fontweight='bold')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
        plt.tight_layout()
        
        # Save
        out_sub = os.path.join(output_dir, analysis, dataset)
        os.makedirs(out_sub, exist_ok=True)
        fname = f"boxplot{'_risk'+str(risk) if analysis == 'Competing_Risks' else ''}.png"
        plt.savefig(os.path.join(out_sub, fname), dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved: {fname}")

    # 5. Display Summary Table 
    print("\n=== DATA SUMMARY ===")
    summary = df_top.groupby(["analysis", "dataset", "risk", "label"])["ci_minus_ibs"].agg(
        Top_Values=lambda x: list(np.round(x, 4)),
        Mean='mean'
    ).reset_index()
    display(summary)

## 6. Generate Plots

In [91]:
output_dir = os.path.abspath(SCRIPT_CONFIG["output_dir"])
plot_boxplots_all_combinations(df, output_dir, 
               datasets=SCRIPT_CONFIG["datasets"], 
               analyses=SCRIPT_CONFIG["analyses"],
               top_n=3)

Saved: boxplot_risk0.png
Saved: boxplot_risk1.png
Saved: boxplot_risk0.png
Saved: boxplot_risk1.png
Saved: boxplot.png
Saved: boxplot.png

=== DATA SUMMARY ===


,analysis,dataset,risk,label,Top_Values,Mean
0,Competing_Risks,brca,0,clinical_10,"[0.424, 0.403, 0.399]",0.408667
1,Competing_Risks,brca,0,clinical_5,"[0.426, 0.395, 0.391]",0.404000
2,Competing_Risks,brca,0,clinical_omic_RNAseq_5,"[0.404, 0.392, 0.374]",0.390000
3,Competing_Risks,brca,0,clinical_omic_RNAseq_50,"[0.413, 0.412, 0.365]",0.396667
4,Competing_Risks,brca,0,clinical_omic_RNAseq_500,"[0.425, 0.388, 0.379]",0.397333
...,...,...,...,...,...,...
97,Survival_Analysis,lgg,0,clinical_omic_miRNA_50,"[0.451, 0.43, 0.385]",0.422000
98,Survival_Analysis,lgg,0,clinical_omic_miRNA_500,"[0.388, 0.385, 0.362]",0.378333
99,Survival_Analysis,lgg,0,clinical_wsi_patches_1_patch_5,"[0.518, 0.485, 0.454]",0.485667
100,Survival_Analysis,lgg,0,clinical_wsi_patches_1_patch_50,"[0.458, 0.418, 0.404]",0.426667
